# logging.torch

> 检查PyTorch模型是否符合预期，是否为要训练的模型

In [ ]:
#| default_exp logging.torch

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2
from nbdev.showdoc import *

In [ ]:
#| export
from scholarly_infrastructure.logging.nucleus import logger

In [ ]:
#| export
from fastcore.basics import patch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from rich.table import Table

@patch
def inspect_model_parameters(model:nn.Module):
    trainable_params = 0
    all_param = 0
    trainable_bytes = 0
    all_bytes = 0
    for _, param in model.named_parameters():
        param_bytes = param.numel() * param.element_size()
        all_param += param.numel()
        all_bytes += param_bytes
        if param.requires_grad:
            trainable_params += param.numel()
            trainable_bytes += param_bytes
    return trainable_params, all_param, trainable_bytes, all_bytes

@patch
def num_of_total_parameters(model:nn.Module):
    return (model).inspect_model_parameters()[1]

@patch
def num_of_trainable_parameters(model:nn.Module):
    return (model).inspect_model_parameters()[0]

@patch
def print_trainable_parameters(model:nn.Module):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params, all_param, trainable_bytes, all_bytes = model.inspect_model_parameters()
    # print(
    table = Table(title=f"Model {model.__class__.__name__}'s Trainable Parameters Inspection")
    table.add_column("Number of Trainable Parameters", justify="right", style="cyan", no_wrap=True)
    table.add_column("Number of Total Parameters", style="magenta")
    table.add_column("Trainable Ratio (0-1)", justify="right", style="green")
    table.add_row(f"{trainable_params:.3e} ({trainable_bytes:.3e} bytes)", f"{all_param:.3e} ({all_bytes:.3e} bytes)", f"{trainable_params / all_param:.3e}")
    
    logger.info(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param:.2f}", 
        rich=table
    )

In [ ]:
from transformers import AutoModel, AutoConfig
test_model = AutoModel.from_config(AutoConfig.from_pretrained("google/vit-base-patch16-224-in21k"))

In [ ]:
test_model.print_trainable_parameters()

In [ ]:
#| export
from bigmodelvis import Visualization
@patch
def model_rich_tree(self:nn.Module):
    module_tree = Visualization(self).structure_graph(printTree=False)
    return module_tree

from rich.panel import Panel
@patch
def print_model_pretty(self:nn.Module):
    module_tree = self.model_rich_tree()
    panel = Panel(module_tree, title=f"Model Tree for {self.__class__.__name__}")
    logger.info(str(self), rich=panel)
    # return module_tree

In [ ]:
test_model.print_model_pretty()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

## What it does

`scholarly_infrastructure.logging.torch` patches `torch.nn.Module` with inspection helpers, so you can sanity-check a model **before** training — called directly on the module instance, with rich output:

- `model.inspect_model_parameters()` -> `(trainable_params, all_param, trainable_bytes, all_bytes)`
- `model.num_of_trainable_parameters()` / `model.num_of_total_parameters()`
- `model.print_trainable_parameters()` -> logs a rich table (counts, bytes, and trainable ratio) through the library logger
- `model.model_rich_tree()` -> a `rich` tree of the module structure (via `bigmodelvis`)
- `model.print_model_pretty()` -> prints that tree

The patches are applied when you import the module, so a plain `import scholarly_infrastructure.logging.torch` (or `import scholarly_infrastructure.logging`) is enough.

### Example

```python
import torch.nn as nn
import scholarly_infrastructure.logging.torch  # applies the patches

model = nn.Transformer(d_model=64, nhead=4, num_encoder_layers=2)
model.print_trainable_parameters()             # rich table: trainable vs total
tree = model.model_rich_tree()                 # a rich tree of the layers
```
